# SafeNight 데이터 병합

2026-11-03 S-DoT 관측, 2026-03-10 센서 위치와 과거 CCTV 자료를 사용했다. 가로등·보안등은 미확보 자료로 남겼다.

[입력 파일·해시](data/merge/manifest.json)

In [21]:
import sys
import os
from pathlib import Path

NOTEBOOK_DIR = Path(os.environ['SAFENIGHT_NOTEBOOK_DIR']) if 'SAFENIGHT_NOTEBOOK_DIR' in os.environ else next(
    path for path in [Path.cwd(), Path.cwd()/'notebooks'] if (path/'data').is_dir()
)
assert sys.version_info >= (3,11)
print(f'Python {sys.version.split()[0]}')


Python 3.11.15


In [22]:
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from IPython.display import display

m_data = NOTEBOOK_DIR / 'data/merge'
m_output = NOTEBOOK_DIR.parent / 'docs/notebook-results'
m_output.mkdir(parents=True, exist_ok=True)
m_manifest = json.loads((m_data/'manifest.json').read_text())
for m_entry in m_manifest:
    assert hashlib.sha256((m_data/m_entry['file']).read_bytes()).hexdigest() == m_entry['snapshot_sha256']
observations = pd.read_csv(m_data/'observations.csv')
Loc = pd.read_csv(m_data/'locations.csv')
df_Cctv = pd.read_csv(m_data/'cctv_legacy.csv')
df_light = None
df_secure = None
analysis_date = '2026-11-03'
radius_m = 25
display(pd.DataFrame(m_manifest)[['file','rows','source_file']])
assert not Loc['sensor_id'].duplicated().any(), '센서 위치 키 중복'
assert not observations.duplicated(['sensor_id','observed_at_kst']).any(), '관측 키 중복'
print('위치 키·관측 키의 유일성 확인')

,file,rows,source_file
0,observations.csv,24715,SDOT_정규화관측_위치결합_2026-11-03.csv
1,locations.csv,1170,센서위치_공식파일_2026-03-10.csv
2,cctv_legacy.csv,59414,CCTV_기존보관자료_2022_신규API아님.csv


위치 키·관측 키의 유일성 확인


## 1. 관측·위치 병합

센서 ID로 위치를 다대일 병합하고, 한국 시각 00~02시·22~23시 관측을 남겼다.

In [23]:
Sdot = observations.rename(columns={
    'MAX_INTE_ILLU':'lux_max', 'AVG_INTE_ILLU':'lux_avg', 'MIN_INTE_ILLU':'lux_min',
    'MAX_NOISE':'noise_max', 'AVG_NOISE':'noise_avg', 'MIN_NOISE':'noise_min',
})
feature_columns = ['lux_max','lux_avg','lux_min','noise_max','noise_avg','noise_min']
Sdot['time'] = pd.to_datetime(Sdot['observed_at_kst'], utc=True).dt.tz_convert('Asia/Seoul')
for column in feature_columns:
    Sdot[column] = pd.to_numeric(Sdot[column], errors='coerce')
m_steps = [('정규화 관측',len(Sdot))]
merged_df = Sdot.merge(Loc, on='sensor_id', how='left', validate='many_to_one', indicator=True)
assert len(merged_df) == len(Sdot), '병합으로 관측 행이 늘어남'
print('위치 미연결', int((merged_df['_merge']!='both').sum()))
merged_df = merged_df[merged_df['time'].dt.strftime('%Y-%m-%d').eq(analysis_date)]
m_steps.append(('분석 날짜',len(merged_df)))
merged_df = merged_df[(merged_df['time'].dt.hour<3) | (merged_df['time'].dt.hour>=22)]
m_steps.append(('야간 5개 시간대',len(merged_df)))
merged_df = merged_df.dropna(subset=['lat','lon',*feature_columns]).copy()
m_steps.append(('필수값·위치 확보',len(merged_df)))
swap = merged_df['lat'].between(124,132) & merged_df['lon'].between(33,39)
merged_df.loc[swap,['lat','lon']] = merged_df.loc[swap,['lon','lat']].to_numpy()
valid = merged_df['lat'].between(33,39) & merged_df['lon'].between(124,132)
print('좌표 교환',int(swap.sum()),'좌표 제외',int((~valid).sum()))
df_Sdot = merged_df.loc[valid,['sensor_id','time','lat','lon',*feature_columns]].copy()
m_steps.append(('국내 좌표 범위',len(df_Sdot)))
assert not df_Sdot.duplicated(['sensor_id','time']).any()
display(pd.DataFrame(m_steps,columns=['단계','행 수']))
display(df_Sdot.head(3))

위치 미연결 981
좌표 교환 0 좌표 제외 0


,단계,행 수
0,정규화 관측,24715
1,분석 날짜,24715
2,야간 5개 시간대,5233
3,필수값·위치 확보,3834
4,국내 좌표 범위,3834


,sensor_id,time,lat,lon,lux_max,lux_avg,lux_min,noise_max,noise_avg,noise_min
0,OC3CL200013,2026-11-03 00:00:00+09:00,37.583469,126.982622,0.0,0.0,0.0,41.0,40.0,40.0
1,OC3CL200022,2026-11-03 00:00:00+09:00,37.503092,127.049876,1.0,1.0,1.0,61.0,57.0,54.0
2,OC3CL200046,2026-11-03 00:00:00+09:00,37.560536,126.854687,1.0,1.0,1.0,43.0,42.0,41.0


## 2. 주변 CCTV 집계

CCTV 원본의 중복 10행을 제거한 자료에서 반경 25m의 기록 수와 카메라 수량을 따로 집계했다.

In [24]:
assert not df_Cctv['source_row'].duplicated().any()
for column in ['위도','경도','CCTV 수량']:
    df_Cctv[column] = pd.to_numeric(df_Cctv[column],errors='coerce')
facility_swap = df_Cctv['위도'].between(124,132) & df_Cctv['경도'].between(33,39)
df_Cctv.loc[facility_swap,['위도','경도']] = df_Cctv.loc[facility_swap,['경도','위도']].to_numpy()
facility_ok = (df_Cctv['위도'].between(33,39) & df_Cctv['경도'].between(124,132)
               & df_Cctv['CCTV 수량'].ge(0) & np.isfinite(df_Cctv['CCTV 수량']))
print('CCTV 좌표 교환',int(facility_swap.sum()),'유효하지 않은 기록',int((~facility_ok).sum()))
df_Cctv = df_Cctv.loc[facility_ok].copy()
positions = df_Sdot[['lat','lon']].drop_duplicates().reset_index(drop=True)
cctv_tree = BallTree(np.radians(df_Cctv[['위도','경도']]), metric='haversine')
neighbors = cctv_tree.query_radius(np.radians(positions[['lat','lon']]), r=radius_m/6371000)
quantity = df_Cctv['CCTV 수량'].to_numpy()
positions['CCTV_기록수_25m'] = [len(ids) for ids in neighbors]
positions['CCTV_수량_25m'] = [float(quantity[ids].sum()) for ids in neighbors]
merged_count = len(df_Sdot)
df_Sdot = df_Sdot.merge(positions,on=['lat','lon'],how='left',validate='many_to_one')
assert len(df_Sdot) == merged_count
assert not df_Sdot[['CCTV_기록수_25m','CCTV_수량_25m']].isna().any().any()
df_Sdot['가로등_자료'] = pd.NA
df_Sdot['보안등_자료'] = pd.NA
display(df_Sdot.head(3))
print('최종 관측',len(df_Sdot),'행 / 센서',df_Sdot['sensor_id'].nunique(),'개 / 좌표',len(positions),'곳')

CCTV 좌표 교환 0 유효하지 않은 기록 0


,sensor_id,time,lat,lon,lux_max,lux_avg,lux_min,noise_max,noise_avg,noise_min,CCTV_기록수_25m,CCTV_수량_25m,가로등_자료,보안등_자료
0,OC3CL200013,2026-11-03 00:00:00+09:00,37.583469,126.982622,0.0,0.0,0.0,41.0,40.0,40.0,3,3.0,<NA>,<NA>
1,OC3CL200022,2026-11-03 00:00:00+09:00,37.503092,127.049876,1.0,1.0,1.0,61.0,57.0,54.0,4,4.0,<NA>,<NA>
2,OC3CL200046,2026-11-03 00:00:00+09:00,37.560536,126.854687,1.0,1.0,1.0,43.0,42.0,41.0,1,1.0,<NA>,<NA>


최종 관측 3834 행 / 센서 947 개 / 좌표 947 곳


## 3. 검증·저장

병합 전후 행 수와 관측 키 중복, 미확보 조명 자료의 결측을 검사했다. [병합 데이터](../docs/notebook-results/merged_observations.csv)와 [검증 기록](../docs/notebook-results/merge-quality.json)을 저장했다.

In [25]:
assert len(df_Sdot) > 0
assert not df_Sdot.duplicated(['sensor_id','time']).any()
assert df_Sdot[['가로등_자료','보안등_자료']].isna().all().all()
assert (df_Sdot[feature_columns] >= 0).all().all()
assert (df_Sdot['noise_min'] <= df_Sdot['noise_avg']).all()
assert (df_Sdot['noise_avg'] <= df_Sdot['noise_max']).all()
m_quality = {'date': analysis_date, 'timezone':'Asia/Seoul', 'night_hours':[0,1,2,22,23],
             'grain':'sensor_id, time', 'rows':len(df_Sdot), 'sensors':int(df_Sdot.sensor_id.nunique()),
             'sites':len(positions), 'radius_m':radius_m, 'unmatched_location_rows':int((Sdot.merge(Loc,on='sensor_id',how='left',validate='many_to_one')['lat'].isna()).sum()),
             'missing_facilities':['streetlights','security_lights'], 'cctv_reference':'legacy_2022_not_current',
             'filter_rows':dict(m_steps), 'units':{'lux':'lx','noise':'dB','distance':'m'}}
df_Sdot.to_csv(m_output/'merged_observations.csv',index=False)
(m_output/'merge-quality.json').write_text(json.dumps(m_quality,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(m_quality,ensure_ascii=False,indent=2))

{
  "date": "2026-11-03",
  "timezone": "Asia/Seoul",
  "night_hours": [
    0,
    1,
    2,
    22,
    23
  ],
  "grain": "sensor_id, time",
  "rows": 3834,
  "sensors": 947,
  "sites": 947,
  "radius_m": 25,
  "unmatched_location_rows": 981,
  "missing_facilities": [
    "streetlights",
    "security_lights"
  ],
  "cctv_reference": "legacy_2022_not_current",
  "filter_rows": {
    "정규화 관측": 24715,
    "분석 날짜": 24715,
    "야간 5개 시간대": 5233,
    "필수값·위치 확보": 3834,
    "국내 좌표 범위": 3834
  },
  "units": {
    "lux": "lx",
    "noise": "dB",
    "distance": "m"
  }
}


## 결과

야간 5,233행 중 위치와 필수 관측값을 갖춘 3,834행·947센서를 남겼다.